In [1]:
!pip install -q boto3 cryptography

In [ ]:
import os
import json
import base64
from dotenv import load_dotenv
import hashlib
import mimetypes
import re
import secrets    ## Used to generate cryptographically secure random AES keys.
import tempfile   ## Creates controlled temporary storage.
from concurrent.futures import ThreadPoolExecutor

from pathlib import Path
from datetime import datetime, timezone

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config

from cryptography.hazmat.primitives import hashes   ## Used for SHA-256 hashing and RSA signature verification.
from cryptography.hazmat.primitives.asymmetric import padding    ## Used for RSA signature verification.
from cryptography.hazmat.primitives.serialization import (
  load_der_public_key,   ## Converts the public key returned by KMS in DER ENCODED FORMAT into a Python RSA public-key object. 
 )

from cryptography.hazmat.primitives.ciphers import (    ## Used for AES-256-GCM.
  Cipher,
  algorithms,
  modes,
 )

In [2]:
load_dotenv()  ## Load environment variables from .env file.

True

In [3]:
AWS_REGION = os.environ["AWS_REGION"]

S3_BUCKET = os.environ["MEDICAL_RAG_BUCKET"]

KMS_ENCRYPTION_KEY_ID = os.environ[
  "KMS_ENCRYPTION_KEY_ID"
]

KMS_SIGNING_KEY_ID = os.environ[
  "KMS_SIGNING_KEY_ID"
]

S3_SSE_KMS_KEY_ID = os.environ[
  "S3_SSE_KMS_KEY_ID"
]

S3_PREFIX = "medical-documents"

In [4]:
AWS_CONFIG = Config(   ## Creates AWS SDK configuration.
  region_name=AWS_REGION,
  retries={
    "max_attempts": 5,
    "mode": "adaptive",
  },
  connect_timeout=10,
  read_timeout=60,
)

session = boto3.Session()   ## Uses the normal AWS credential chain.

kms = session.client(   ## Creates the KMS client.
  "kms",
  config=AWS_CONFIG,
)

s3 = session.client(   ## Creates the S3 client.
  "s3",
  config=AWS_CONFIG,
)

print("AWS clients initialized.")

AWS clients initialized.


In [5]:
def generate_document_id() -> str:
  return secrets.token_hex(16)

In [6]:
generate_document_id()

'745838706aed60430edc0f8320cbae90'

In [7]:
USER_ID_PATTERN = re.compile(
    r"^[A-Za-z0-9_-]{1,128}$"
)

DOCUMENT_ID_PATTERN = re.compile(
    r"^[a-f0-9]{32}$"
)


def validate_identifiers(
    user_id: str,
    document_id: str,
) -> None:

    if not USER_ID_PATTERN.fullmatch(user_id):
        raise ValueError("Invalid user_id.")

    if not DOCUMENT_ID_PATTERN.fullmatch(document_id):
        raise ValueError("Invalid document_id.")

In [8]:
def get_rsa_public_key(kms_client, kms_key_id: str):   ## Retrieves the RSA public key from AWS KMS.
  response = kms_client.get_public_key(
    KeyId=kms_key_id
  )

  public_key = load_der_public_key(
    response["PublicKey"]
  )

  return public_key

In [9]:
get_rsa_public_key(kms, KMS_ENCRYPTION_KEY_ID)

In [10]:
def generate_aes_key() -> bytes:   ## Generates a random 256-bit AES key using the secrets module for cryptographic security.
  return secrets.token_bytes(32)

In [11]:
generate_aes_key()

b"L\xd212'\xb9\x0ev'\x84\x83\x99\xa7k\x8a\xc7\xc7\xc0|v\xca\xca\x90dz\x08\xd0\xa4\x15\x99\x04\xb1"

In [12]:
def encrypt_file(   ## Encrypts a file using AES-256-GCM and writes the encrypted data to a new file.
  input_path: str,
  output_path: str,
  aes_key: bytes,
):
  if len(aes_key) != 32:
    raise ValueError(
      "AES key must be exactly 32 bytes."
    )

  nonce = secrets.token_bytes(12)  ## GCM uses a 12-byte nonce.

  encryptor = Cipher(
    algorithms.AES(aes_key),    ## The actual mathematical algorithm for encyption.  Block cipher
    modes.GCM(nonce),     ## This is the mode of operation for AES which tells how to handle many blocks as its a block cipher. It requires a nonce (number used once) to ensure that the same plaintext encrypts to different ciphertexts each time.
  ).encryptor()

  ciphertext_hash = hashlib.sha256()

  with (
    open(input_path, "rb") as source,
    open(output_path, "wb") as target,
  ):
    while True:
      chunk = source.read(1024 * 1024)   ## Read in 1 MB chunks for efficiency.

      if not chunk:
        break

      encrypted_chunk = encryptor.update(
        chunk
      )

      target.write(encrypted_chunk)

      ciphertext_hash.update(
        encrypted_chunk
      )

    encryptor.finalize()  ## Finalize the encryption process to ensure all data is processed.

    tag = encryptor.tag   ## The GCM tag allows us to detect tampering.

  return {
    "nonce": nonce,
    "tag": tag,
    "ciphertext_sha256": ciphertext_hash.hexdigest(),
  }

### Encrypt the AES key using the RSA public key
---
*Specifies that OAEP (Optimal Asymmetric Encryption Padding) must be used. Raw RSA is mathematically vulnerable; padding adds structured random data to the payload before encryption. This ensures that encrypting the same AES key twice results in two completely different ciphertexts, preventing attackers from guessing the key.*
>
*• Defines the Mask Generation Function (MGF1), which is an internal component of OAEP used to generate a stream of pseudorandom bits. It is configured here to use the secure SHA-256 hashing algorithm to produce those bits.*

In [13]:
def encrypt_aes_key_with_rsa(
  aes_key: bytes,
  public_key,  ## Public key for the encryption decryption KMS
) -> bytes:
  encrypted_aes_key = public_key.encrypt(    ## .encrypt() method encrypts the AES key using RSA public key with OAEP padding and SHA-256 hashing.
    aes_key,
    padding.OAEP(
      mgf=padding.MGF1(
        algorithm=hashes.SHA256()
      ),
      algorithm=hashes.SHA256(),
      label=None,
    ),
  )

  return encrypted_aes_key

#### Build The Manifest

In [14]:
def canonical_json(data: dict) -> bytes:   ## Converts a Python dictionary into a canonical JSON byte string for consistent hashing and signing. It will sort the keys and use compact separators to ensure that the JSON representation is consistent across different platforms and languages.
  return json.dumps(                       ## It will also help in generating a consistent hash for the manifest, which is crucial for signature verification.
    data,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
  ).encode("utf-8")

In [15]:
def build_manifest(
  *,
  document_id: str,
  user_id: str,
  original_filename: str,
  mime_type: str,
  original_size: int,
  encrypted_aes_key: bytes,
  nonce: bytes,
  tag: bytes,
  ciphertext_sha256: str,
):
  return {
    "format_version": 1,

    "document_id": document_id,

    "user_id": user_id,

    "document": {
      "filename": original_filename,
      "mime_type": mime_type,
      "original_size": original_size,
    },

    "encryption": {
      "document_algorithm": "AES-256-GCM",

      "key_algorithm": "RSA-OAEP-SHA256",

      "encryption_key_id": KMS_ENCRYPTION_KEY_ID,

      "encrypted_aes_key": (
        base64.b64encode(    ## encrypted_aes_key is bytes form. base64.b64encode() converts it to base64 bytes, and .decode("ascii") converts it to a string for JSON serialization.
          encrypted_aes_key
        ).decode("ascii")
      ),

      "nonce": (
        base64.b64encode(
          nonce
        ).decode("ascii")
      ),

      "tag": (
        base64.b64encode(
          tag
        ).decode("ascii")
      ),
    },

    "ciphertext": {
      "sha256": ciphertext_sha256,
    },

    "created_at": datetime.now(
      timezone.utc
    ).isoformat(),
  }

#### Sign the Manifest Using the KMS Private Key for the Digital Signature
*Computes the SHA-256 hash of the canonical JSON representation of the manifest. This digest is what will be signed by KMS to ensure the integrity and authenticity of the manifest.*

*Then encrypts the hashed digest using the Private key.*

In [16]:
def sign_manifest(
  kms_client,
  signing_key_id: str,
  manifest: dict,
):
  manifest_bytes = canonical_json(manifest)

  digest = hashlib.sha256(
    manifest_bytes
  ).digest()

  response = kms_client.sign(
    KeyId=signing_key_id,

    Message=digest,

    MessageType="DIGEST",  ## Indicates that the message is already hashed, so KMS will not hash it again.

    SigningAlgorithm="RSASSA_PSS_SHA_256",
  )

  return response["Signature"]

#### Create temporary storage  and the whole function to prepare the document

In [17]:
temporary_directory = tempfile.TemporaryDirectory(
  prefix="medical_rag_"
)

TEMP_DIR = Path(
  temporary_directory.name
)

print(
  "Temporary workspace:",
  TEMP_DIR
)

Temporary workspace: C:\Users\rajde\AppData\Local\Temp\medical_rag_o9zp0f_s


In [18]:
def prepare_document(
  *,
  input_path: str,
  user_id: str,
  document_id: str,
):
  input_file = Path(input_path)

  if not input_file.is_file():
    raise FileNotFoundError(
      f"File not found: {input_path}"
    )

  validate_identifiers(
    user_id,
    document_id,
  )

  mime_type, _ = mimetypes.guess_type(
    input_file.name
  )

  if mime_type is None:
    mime_type = (
      "application/octet-stream"
    )

  encrypted_path = (
    TEMP_DIR /
    f"{document_id}.ciphertext"
  )

  # --------------------------------------------
  # 1. Get public encryption key from KMS
  # --------------------------------------------

  encryption_public_key = (
    get_rsa_public_key(
      kms,
      KMS_ENCRYPTION_KEY_ID,
    )
  )

  # --------------------------------------------
  # 2. Generate unique AES key
  # --------------------------------------------

  aes_key = generate_aes_key()

  try:
    # ----------------------------------------
    # 3. Encrypt actual document with AES
    # ----------------------------------------

    encryption_result = encrypt_file(
      input_path=str(input_file),

      output_path=str(
        encrypted_path
      ),

      aes_key=aes_key,
    )

    # ----------------------------------------
    # 4. Protect AES key with RSA public key
    # ----------------------------------------

    encrypted_aes_key = (
      encrypt_aes_key_with_rsa(
        aes_key=aes_key,
        public_key=encryption_public_key,
      )
    )

    # ----------------------------------------
    # 5. Build manifest
    # ----------------------------------------

    manifest = build_manifest(
      document_id=document_id,

      user_id=user_id,

      original_filename=input_file.name,

      mime_type=mime_type,

      original_size=input_file.stat().st_size,

      encrypted_aes_key=encrypted_aes_key,

      nonce=encryption_result["nonce"],

      tag=encryption_result["tag"],

      ciphertext_sha256=(
        encryption_result[
          "ciphertext_sha256"
        ]
      ),
    )

    # ----------------------------------------
    # 6. Sign manifest with KMS private key
    # ----------------------------------------

    signature = sign_manifest(
      kms_client=kms,

      signing_key_id=(
        KMS_SIGNING_KEY_ID
      ),

      manifest=manifest,
    )

    return {
      "encrypted_path": encrypted_path,

      "manifest": manifest,

      "signature": signature,
    }

  finally:
    # ----------------------------------------
    # 7. Remove AES key reference
    # ----------------------------------------

    del aes_key

#### Upload the encrypted file to S3

In [ ]:
def upload_document(
    *,
    prepared: dict,
    bucket: str,
    user_id: str,
    document_id: str,
 ):
    encrypted_path = (
        prepared["encrypted_path"]
    )

    manifest = prepared["manifest"]

    signature = prepared["signature"]

    prefix = (
        f"{S3_PREFIX}/"
        f"{user_id}/"
        f"{document_id}"
    )

    ciphertext_key = (
        f"{prefix}/ciphertext.bin"
    )

    signature_key = (
        f"{prefix}/signature.bin"
    )

    manifest_key = (
        f"{prefix}/manifest.json"
    )

    transfer_config = TransferConfig(
        multipart_threshold=8 * 1024 * 1024,
        multipart_chunksize=8 * 1024 * 1024,
        max_concurrency=4,
        use_threads=True,
    )

    uploaded_keys = []

    def upload_ciphertext():
        s3.upload_file(
            str(encrypted_path),
            bucket,
            ciphertext_key,
            ExtraArgs={
                "ContentType": "application/octet-stream",
                "ServerSideEncryption": "aws:kms",
                "SSEKMSKeyId": S3_SSE_KMS_KEY_ID,
                "BucketKeyEnabled": True,
            },
            Config=transfer_config,
        )
        return ciphertext_key

    def upload_signature():
        s3.put_object(
            Bucket=bucket,
            Key=signature_key,
            Body=signature,
            ContentType="application/octet-stream",
            ServerSideEncryption="aws:kms",
            SSEKMSKeyId=S3_SSE_KMS_KEY_ID,
            BucketKeyEnabled=True,
        )
        return signature_key

    try:
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(upload_ciphertext),
                executor.submit(upload_signature),
            ]

            upload_errors = []
            for future in futures:
                try:
                    uploaded_keys.append(future.result())
                except Exception as error:
                    upload_errors.append(error)

            if upload_errors:
                raise upload_errors[0]

        manifest_bytes = canonical_json(manifest)

        s3.put_object(
            Bucket=bucket,
            Key=manifest_key,
            Body=manifest_bytes,
            ContentType="application/json",
            ServerSideEncryption="aws:kms",
            SSEKMSKeyId=S3_SSE_KMS_KEY_ID,
            BucketKeyEnabled=True,
        )

        uploaded_keys.append(manifest_key)

        return {
            "bucket": bucket,
            "document_id": document_id,
            "ciphertext_key": ciphertext_key,
            "manifest_key": manifest_key,
            "signature_key": signature_key,
        }

    except Exception:
        for key in uploaded_keys:
            try:
                s3.delete_object(
                    Bucket=bucket,
                    Key=key,
                )
            except Exception:
                pass

        raise

In [20]:
INPUT_FILE = "./RAJDEEP_PAUL_IMAGE.jpg"

USER_ID = "user_123"

DOCUMENT_ID = generate_document_id()

print("Document ID:")
print(DOCUMENT_ID)

prepared = prepare_document(
  input_path=INPUT_FILE,

  user_id=USER_ID,

  document_id=DOCUMENT_ID,
)

print()
print("Encryption completed.")
print("Signature generated.")

Document ID:
cb809f492c08af989437944ee8e52169

Encryption completed.
Signature generated.


In [21]:
upload_document(
  prepared=prepared,
  bucket=S3_BUCKET,
  user_id=USER_ID,
  document_id=DOCUMENT_ID
)

{'bucket': 'medical-rag-secure-documents',
 'document_id': 'cb809f492c08af989437944ee8e52169',
 'ciphertext_key': 'medical-documents/user_123/cb809f492c08af989437944ee8e52169/ciphertext.bin',
 'manifest_key': 'medical-documents/user_123/cb809f492c08af989437944ee8e52169/manifest.json',
 'signature_key': 'medical-documents/user_123/cb809f492c08af989437944ee8e52169/signature.bin'}

#### Decryption

In [22]:
from cryptography.exceptions import InvalidSignature, InvalidTag

In [ ]:
def download_encrypted_package(
  *,
  bucket: str,
  user_id: str,
  document_id: str,
  output_directory: str,
 ):
    prefix = f"{S3_PREFIX}/{user_id}/{document_id}"

    ciphertext_key = f"{prefix}/ciphertext.bin"
    manifest_key = f"{prefix}/manifest.json"
    signature_key = f"{prefix}/signature.bin"

    output_dir = Path(output_directory)
    output_dir.mkdir(parents=True, exist_ok=True)

    ciphertext_path = output_dir / "ciphertext.bin"
    manifest_path = output_dir / "manifest.json"
    signature_path = output_dir / "signature.bin"

    transfer_config = TransferConfig(
        multipart_threshold=8 * 1024 * 1024,
        multipart_chunksize=8 * 1024 * 1024,
        max_concurrency=4,
        use_threads=True,
    )

    downloads = [
        (ciphertext_key, ciphertext_path),
        (manifest_key, manifest_path),
        (signature_key, signature_path),
    ]

    def download_one(download):
        object_key, destination = download
        s3.download_file(
            bucket,
            object_key,
            str(destination),
            Config=transfer_config,
        )

    # Download all package objects concurrently. Each large object may also
    # use multipart worker threads through TransferConfig.
    with ThreadPoolExecutor(max_workers=3) as executor:
        list(executor.map(download_one, downloads))

    return {
        "ciphertext_path": ciphertext_path,
        "manifest_path": manifest_path,
        "signature_path": signature_path,
    }

In [25]:
def load_manifest(manifest_path: str) -> dict:   ## Converts the manifest JSON file into a Python dictionary for further processing.
    with open(manifest_path, "rb") as file:
        manifest_bytes = file.read()

    return json.loads(
        manifest_bytes.decode("utf-8")
    )

##### Verify the digital signature

In [34]:
from cryptography.hazmat.primitives.asymmetric import utils
def verify_manifest_signature(
    *,
    kms_client,
    signing_key_id: str,
    manifest: dict,
    signature: bytes,
) -> bool:
    # Convert manifest to the exact canonical representation
    manifest_bytes = canonical_json(manifest)

    # Hash the manifest
    digest = hashlib.sha256(manifest_bytes).digest()

    # Get the PUBLIC key from AWS KMS
    response = kms_client.get_public_key(
        KeyId=signing_key_id
    )

    public_key = load_der_public_key(
        response["PublicKey"]
    )

    try:
        # Verify the signature
        public_key.verify(
            signature,
            digest,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=hashes.SHA256().digest_size,
            ),
            utils.Prehashed(hashes.SHA256()),    ## DO NOT hash the digest again, as it is already hashed. This is important because KMS uses the RSASSA_PSS_SHA_256 algorithm, which expects a pre-hashed message.
        )

        return True

    except InvalidSignature:
        return False

##### Is the downloaded ciphertext.bin exactly the ciphertext that was originally uploaded?

In [27]:
def verify_ciphertext_hash(
    *,
    ciphertext_path: str,
    expected_hash: str,
) -> bool:

    sha256 = hashlib.sha256()

    with open(ciphertext_path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    actual_hash = sha256.hexdigest()

    return actual_hash == expected_hash

##### Use the private RSA key associated with this KMS key to decrypt this encrypted AES key.

In [28]:
def decrypt_aes_key(
    *,
    kms_client,
    kms_key_id: str,
    encrypted_aes_key: bytes,
) -> bytes:

    response = kms_client.decrypt(
        KeyId=kms_key_id,
        CiphertextBlob=encrypted_aes_key,
        EncryptionAlgorithm="RSAES_OAEP_SHA_256",
    )

    return response["Plaintext"]

#### Extract encryption information from manifest

In [29]:
def extract_crypto_values(manifest: dict):
    encryption = manifest["encryption"]

    encrypted_aes_key = base64.b64decode(
        encryption["encrypted_aes_key"]
    )

    nonce = base64.b64decode(   ## Decodes the base64-encoded ascii string back into bytes for use in AES-GCM decryption.
        encryption["nonce"]
    )

    tag = base64.b64decode(
        encryption["tag"]
    )

    expected_hash = manifest["ciphertext"]["sha256"]

    return {
        "encrypted_aes_key": encrypted_aes_key,
        "nonce": nonce,
        "tag": tag,
        "expected_hash": expected_hash,
    }

#### AES-256-GCM decryption

In [30]:
def decrypt_file(
    *,
    ciphertext_path: str,
    output_path: str,
    aes_key: bytes,
    nonce: bytes,
    tag: bytes,
):
    if len(aes_key) != 32:
        raise ValueError(
            "AES key must be exactly 32 bytes."
        )

    decryptor = Cipher(
        algorithms.AES(aes_key),
        modes.GCM(nonce, tag),
    ).decryptor()

    with open(ciphertext_path, "rb") as source, \
        open(output_path, "wb") as target:

        while True:
            chunk = source.read(1024 * 1024)

            if not chunk:
                break

            plaintext_chunk = decryptor.update(chunk)

            target.write(plaintext_chunk)

        # IMPORTANT:
        # This verifies the GCM authentication tag.
        decryptor.finalize()

#### Complete Verify + Decrypt Function

In [31]:
class SecurityError(Exception):
  pass

In [32]:
def verify_and_decrypt_document(
  *,
  package: dict,
  output_path: str,
  user_id: str,
  document_id: str,
):
  ciphertext_path = package["ciphertext_path"]
  manifest_path = package["manifest_path"]
  signature_path = package["signature_path"]

  # ---------------------------------------------------------
  # STEP 1: Load manifest
  # ---------------------------------------------------------

  manifest = load_manifest(
    str(manifest_path)
  )

  # ---------------------------------------------------------
  # STEP 2: Check document ID
  # ---------------------------------------------------------

  if manifest["document_id"] != document_id:
    raise SecurityError(
      "Document ID mismatch."
    )

  # ---------------------------------------------------------
  # STEP 3: Check user ID
  # ---------------------------------------------------------

  if manifest["user_id"] != user_id:
    raise SecurityError(
      "User ID mismatch."
    )

  print("✓ Document and user IDs verified.")

  # ---------------------------------------------------------
  # STEP 4: Read digital signature
  # ---------------------------------------------------------

  with open(signature_path, "rb") as file:
    signature = file.read()

  # ---------------------------------------------------------
  # STEP 5: Verify manifest signature
  # ---------------------------------------------------------

  signature_valid = verify_manifest_signature(
    kms_client=kms,
    signing_key_id=KMS_SIGNING_KEY_ID,
    manifest=manifest,
    signature=signature,
  )

  if not signature_valid:
    raise SecurityError(
      "Digital signature verification failed."
    )

  print("✓ Digital signature verified.")

  # ---------------------------------------------------------
  # STEP 6: Extract cryptographic values
  # ---------------------------------------------------------

  crypto = extract_crypto_values(
    manifest
  )

  # ---------------------------------------------------------
  # STEP 7: Verify ciphertext SHA-256
  # ---------------------------------------------------------

  hash_valid = verify_ciphertext_hash(
    ciphertext_path=str(ciphertext_path),
    expected_hash=crypto["expected_hash"],
  )

  if not hash_valid:
    raise SecurityError(
      "Ciphertext SHA-256 verification failed."
  )

  print("✓ Ciphertext SHA-256 verified.")

  # ---------------------------------------------------------
  # STEP 8: Recover AES key using KMS
  # ---------------------------------------------------------

  aes_key = decrypt_aes_key(
    kms_client=kms,
    kms_key_id=KMS_ENCRYPTION_KEY_ID,
    encrypted_aes_key=crypto["encrypted_aes_key"],
  )

  try:
    print(
      "✓ AES key recovered through KMS."
    )

    # -----------------------------------------------------
    # STEP 9: Decrypt document using AES-256-GCM
    # -----------------------------------------------------

    decrypt_file(
      ciphertext_path=str(ciphertext_path),
      output_path=output_path,
      aes_key=aes_key,
      nonce=crypto["nonce"],
      tag=crypto["tag"],
    )

    print(
      "✓ AES-256-GCM authentication verified."
    )

  except InvalidTag:
    raise SecurityError(
      "AES-GCM authentication failed. "
      "The ciphertext, AES key, nonce or tag "
      "may have been modified."
    )

  finally:
    # Remove AES key reference from Python
    del aes_key

  print("✓ Document successfully decrypted.")

  return output_path

In [38]:
package = download_encrypted_package(
  bucket=S3_BUCKET,
  user_id=USER_ID,
  document_id=DOCUMENT_ID,
  output_directory=str(TEMP_DIR),
)

out_path = verify_and_decrypt_document(
  package=package,
  output_path=str(TEMP_DIR / "decrypted_document"),
  user_id=USER_ID,
  document_id=DOCUMENT_ID,
)

✓ Document and user IDs verified.
✓ Digital signature verified.
✓ Ciphertext SHA-256 verified.
✓ AES key recovered through KMS.
✓ AES-256-GCM authentication verified.
✓ Document successfully decrypted.


In [37]:
out_path

'C:\\Users\\rajde\\AppData\\Local\\Temp\\medical_rag_o9zp0f_s\\decrypted_document'